# strip_01 fibroblast subtypes and the immune-fibrosis circuit

I want to push past the coarse `fibroblast` label from notebook 04 and ask a
sharper question: do the IgG-producing cells sit next to a specific *fibrogenic*
fibroblast state, or just generic fibroblasts? If they cluster near
myofibroblast / TGFb-high / hypoxic fibroblasts, that points at an
immune-fibrosis circuit driving keloid pathology.

Plan:
1. Load `adata_annotated.h5ad` from notebook 04.
2. Subset to the 4140 fibroblasts and re-do HVG + PCA + neighbours + Leiden so
   the embedding is driven by fibroblast-internal variation, not by epidermal
   or immune signal that dominated the strip-wide PCA.
3. Score each fibroblast against three fibrogenic gene panels: activated /
   myofibroblast, TGFb signalling, hypoxia.
4. Assign honest sub-labels by combining sub-cluster identity with panel scores.
5. Build a finer label vector for the whole strip, fibroblast sub-labels in place
   of the old `fibroblast` label, everything else preserved.
6. Re-run the permutation neighbourhood enrichment (K=10, 500 perms, two-sided,
   BH-FDR) and read off the IgG_producing row.

I use the same neighbourhood-enrichment function as notebook 04 so results are
directly comparable.


## 0. Setup

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

ROI = 'strip_01'
OUT = Path(f'../../outputs/{ROI}')
FIG = OUT / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

# Subclustering parameters. Fibroblast-only re-embedding.
N_HVG_FIB        = 2000          # tighter than 3000 used strip-wide, since we have 4140 cells
N_PCS_FIB        = 30
N_NEIGHBOURS_FIB = 15
LEIDEN_RES_FIB   = 0.5           # I check 0.3 and 0.8 too, this is the headline

# Same neighbourhood parameters as notebook 04 for a like-for-like comparison.
K_NEIGHBOUR      = 10
N_PERM           = 500
ALPHA            = 0.05

PLASMA_LABEL = 'IgG_producing'

# Reproducibility.
SEED = 0
np.random.seed(SEED)
sc.settings.verbosity = 1


## 1. Load the annotated AnnData from notebook 04

In [ ]:
ANN = OUT / 'adata_annotated.h5ad'
a = ad.read_h5ad(ANN)
print('loaded', a.shape, 'from', ANN)
print()
print('celltype_hc (strict labels from notebook 04):')
print(a.obs['celltype_hc'].value_counts())


I keep `celltype_hc` as the reference labelling. The 93 `IgG_producing` cells
are the strict definition (sub-cluster AND IGHG1>0), which is what I trust for
neighbourhood claims.


## 2. Subset to fibroblasts and re-embed

In [ ]:
fib = a[a.obs['celltype_hc'] == 'fibroblast'].copy()
print('fibroblast subset:', fib.shape)

# AnnData carries log-normalised X from notebook 04. For HVG re-selection on the
# subset I use Seurat flavour on log-normalised data, which is what we used at
# the strip-wide step. No re-normalisation needed since cells were already
# normalised on the whole strip and that scaling is consistent across subsets.

# Re-select HVGs from fibroblast-internal variation. Genes that vary across
# epidermis vs dermis are not interesting here.
sc.pp.highly_variable_genes(fib, n_top_genes=N_HVG_FIB, flavor='seurat')
print('HVGs picked:', int(fib.var['highly_variable'].sum()))

# Drop the obvious structural / housekeeping families so PCs are driven by
# fibroblast state, not by collagen abundance or ribosomal load.
STRUCTURAL_REGEX = r'^(COL\d|KRT\d|MT-|RP[SL]\d|HB[^P])'
struct_mask = fib.var_names.str.match(STRUCTURAL_REGEX)
fib.var.loc[struct_mask, 'highly_variable'] = False
print('after structural drop, HVGs left:', int(fib.var['highly_variable'].sum()))


In [ ]:
# PCA -> neighbours -> UMAP -> Leiden on fibroblast subset only.
sc.tl.pca(fib, n_comps=N_PCS_FIB, mask_var='highly_variable', random_state=SEED)
sc.pp.neighbors(fib, n_neighbors=N_NEIGHBOURS_FIB, n_pcs=N_PCS_FIB, random_state=SEED)
sc.tl.umap(fib, random_state=SEED)

# Three resolutions so I can pick the one that gives interpretable subtypes.
for res in (0.3, 0.5, 0.8):
    sc.tl.leiden(fib, resolution=res, key_added=f'fib_leiden_{res}', random_state=SEED)

for res in (0.3, 0.5, 0.8):
    print(f'res={res}:', fib.obs[f'fib_leiden_{res}'].value_counts().to_dict())


In [ ]:
# UMAP of the three resolutions side by side so I can pick.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, res in zip(axes, (0.3, 0.5, 0.8)):
    sc.pl.umap(fib, color=f'fib_leiden_{res}', ax=ax, show=False,
               legend_loc='on data', frameon=False, title=f'Leiden res={res}')
plt.tight_layout()
plt.savefig(FIG / 'fib_umap_leiden_sweep.png', dpi=150, bbox_inches='tight')
plt.show()


I take the middle resolution (0.5) as the headline. It is granular enough to
separate biological states (a few hundred cells per cluster), and not so fine
that I am splitting noise.


In [ ]:
FIB_LEIDEN = f'fib_leiden_{LEIDEN_RES_FIB}'
fib.obs['fib_subcluster'] = fib.obs[FIB_LEIDEN].astype(str)
print(fib.obs['fib_subcluster'].value_counts())


## 3. Marker genes per fibroblast sub-cluster

In [ ]:
# Rank genes inside the fibroblast subset. These markers are *within-fibroblast*
# markers, i.e. what distinguishes one fibroblast state from another, not from
# epidermis. That is the right reference for sub-typing.
sc.tl.rank_genes_groups(fib, groupby='fib_subcluster', method='wilcoxon',
                        n_genes=20)
# rank_genes_groups stores names as a structured ndarray; column dtype names
# are the group labels.
groups = list(fib.uns['rank_genes_groups']['names'].dtype.names)
markers = pd.DataFrame({g: list(fib.uns['rank_genes_groups']['names'][g])[:15]
                        for g in groups})
print('top markers per sub-cluster:')
markers


## 4. Score fibrogenic gene panels

In [ ]:
# Three panels for the keloid biology question. I score each cell on each panel
# with sc.tl.score_genes, which computes mean expression of the panel minus the
# mean of a matched random reference set. Robust to background expression.

PANELS = {
    'myofibroblast': ['ACTA2', 'TAGLN', 'MYH11', 'POSTN', 'COL1A1', 'COL3A1'],
    'tgfb':          ['TGFB1', 'TGFBR1', 'TGFBR2', 'SMAD3', 'SERPINE1', 'CTGF', 'CCN2'],
    'hypoxia':       ['HIF1A', 'VEGFA', 'LDHA', 'SLC2A1'],
}

# Keep only genes present in the AnnData var.
PANELS = {name: [g for g in genes if g in fib.var_names] for name, genes in PANELS.items()}
for name, genes in PANELS.items():
    print(f'{name}: {len(genes)} genes -> {genes}')

for name, genes in PANELS.items():
    if len(genes) == 0:
        print(f'WARN: no genes for {name}, skipping')
        continue
    sc.tl.score_genes(fib, gene_list=genes, score_name=f'score_{name}',
                      random_state=SEED)


In [ ]:
# UMAP coloured by each score so I can see which sub-cluster lights up.
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
sc.pl.umap(fib, color='fib_subcluster', ax=axes[0], show=False,
           legend_loc='on data', frameon=False, title='sub-cluster')
for ax, name in zip(axes[1:], ['myofibroblast', 'tgfb', 'hypoxia']):
    sc.pl.umap(fib, color=f'score_{name}', ax=ax, show=False,
               frameon=False, cmap='magma', title=name)
plt.tight_layout()
plt.savefig(FIG / 'fib_umap_panel_scores.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Per-sub-cluster mean score table. This is what I use to assign sub-labels.
score_cols = [f'score_{n}' for n in ('myofibroblast', 'tgfb', 'hypoxia')]
mean_scores = fib.obs.groupby('fib_subcluster')[score_cols].mean().round(3)
n_cells = fib.obs['fib_subcluster'].value_counts().rename('n_cells')
panel_table = mean_scores.join(n_cells)
panel_table = panel_table.sort_values('score_myofibroblast', ascending=False)
print('mean panel score per sub-cluster (sorted by myofibroblast):')
panel_table


**Reading the table.** I pick a sub-label per cluster by which panel score is
highest *and* clearly above the across-cluster mean. Clusters that score near
zero on all three are best left as `fibroblast_other` (a quiescent / generic
fibroblast state), since the keloid panels do not pull them up.


In [ ]:
# Threshold-based labelling. For each sub-cluster, take its mean score on each
# panel and compare to the cross-cluster mean. A panel "wins" the cluster only
# if (a) it is the top-scoring panel for that cluster and (b) it is above the
# overall mean for that panel. This guards against assigning a sub-label when
# all panel scores are weak.

panel_means = mean_scores.mean(axis=0)
print('cross-cluster mean per panel:')
print(panel_means.round(3))

label_map = {}
for sub in mean_scores.index:
    row = mean_scores.loc[sub]
    top_panel = row.idxmax().replace('score_', '')
    top_value = row.max()
    if top_value <= panel_means[f'score_{top_panel}']:
        label_map[sub] = 'fibroblast_other'
    else:
        label_map[sub] = f'fib_{top_panel}'

print()
print('sub-cluster -> sub-label:')
for k, v in sorted(label_map.items()):
    print(f'  {k}: {v}  (n={int(n_cells.loc[k])})')

fib.obs['fib_sublabel'] = fib.obs['fib_subcluster'].map(label_map).astype('category')
print()
print('fib_sublabel distribution:')
print(fib.obs['fib_sublabel'].value_counts())


## 5. Spatial map of fibroblast subtypes with IgG-producing cells overlaid

In [ ]:
# Push fib_sublabel back to the full AnnData so I can plot spatial.
a.obs['fib_sublabel'] = pd.Series('NA', index=a.obs_names, dtype=object)
a.obs.loc[fib.obs_names, 'fib_sublabel'] = fib.obs['fib_sublabel'].astype(str).values

# Build the fine label vector. Everything that was 'fibroblast' becomes its
# sub-label, everything else (deep_dermal_stromal, IgG_producing, keratinocytes,
# vascular) is preserved.
fine = a.obs['celltype_hc'].astype(str).copy()
is_fib = fine == 'fibroblast'
fine[is_fib] = a.obs.loc[is_fib, 'fib_sublabel'].astype(str)
a.obs['celltype_fine'] = pd.Categorical(fine)
print('celltype_fine:')
print(a.obs['celltype_fine'].value_counts())


In [ ]:
# Spatial plot. Background = sub-labelled fibroblasts and other cells in light
# greys / colours; foreground = IgG_producing cells as red dots so the question
# (which fibroblast subtype do they sit next to) is visually obvious.

xy = a.obsm['spatial']
fig, ax = plt.subplots(figsize=(14, 5))

# Colour scheme: fibrogenic sub-labels coloured, others grey.
palette = {
    'fib_myofibroblast':  '#d62728',  # red
    'fib_tgfb':           '#9467bd',  # purple
    'fib_hypoxia':        '#ff7f0e',  # orange
    'fibroblast_other':   '#bcbd22',  # olive
    'deep_dermal_stromal': '#8c8c8c',
    'keratinocyte_basal_mix': '#cccccc',
    'keratinocyte_suprabasal': '#dddddd',
    'vascular_mixed':     '#1f77b4',
    'IgG_producing':      '#000000',
}

for lbl, colour in palette.items():
    m = a.obs['celltype_fine'].astype(str).values == lbl
    if m.sum() == 0: continue
    size = 18 if lbl == 'IgG_producing' else 3
    edge = 'white' if lbl == 'IgG_producing' else 'none'
    ax.scatter(xy[m, 0], xy[m, 1], s=size, c=colour, label=f'{lbl} (n={int(m.sum())})',
               linewidths=0.4, edgecolors=edge, alpha=0.95 if lbl == 'IgG_producing' else 0.6)

ax.set_aspect('equal')
ax.invert_yaxis()  # image coordinates
ax.set_xlabel('x (px)')
ax.set_ylabel('y (px)')
ax.set_title('strip_01: fibroblast subtypes with IgG-producing cells overlaid')
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig(FIG / 'fib_subtypes_spatial.png', dpi=180, bbox_inches='tight')
plt.show()


## 6. Neighbourhood enrichment with the finer fibroblast labels

In [ ]:
# Same permutation test as notebook 04. Two-sided p-values, BH-FDR on the full
# label x label matrix, K=10, 500 permutations.

def benjamini_hochberg(pvals):
    p = np.asarray(pvals); shape = p.shape
    flat = p.flatten(); n = len(flat)
    order = np.argsort(flat); ranks = np.arange(1, n + 1)
    sorted_p = flat[order]; raw_adj = sorted_p * n / ranks
    adj_sorted = np.minimum.accumulate(raw_adj[::-1])[::-1]
    adj_sorted = np.minimum(adj_sorted, 1.0)
    adj = np.empty(n); adj[order] = adj_sorted
    return adj.reshape(shape)


def neighbourhood_enrichment(positions, labels, K=10, n_perm=500, seed=0,
                             alternative='two-sided'):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    nn = NearestNeighbors(n_neighbors=K + 1).fit(positions)
    _, idx = nn.kneighbors(positions); idx = idx[:, 1:]
    types = np.unique(labels); nT = len(types)

    def composition(lbls):
        nb = lbls[idx]
        comp = np.zeros((nT, nT))
        for i, a_t in enumerate(types):
            focal = lbls == a_t
            if focal.sum() == 0: continue
            for j, b_t in enumerate(types):
                comp[i, j] = (nb[focal] == b_t).mean()
        return comp

    observed = composition(labels)
    expected = np.tile(
        pd.Series(labels).value_counts(normalize=True).reindex(types).fillna(0).values,
        (nT, 1)
    )

    null = np.zeros((n_perm, nT, nT))
    for p in range(n_perm):
        null[p] = composition(rng.permutation(labels))

    null_mean = null.mean(axis=0)
    null_std  = null.std(axis=0) + 1e-9
    zscore = (observed - null_mean) / null_std

    if alternative == 'two-sided':
        p_hi = ((null >= observed[None]).sum(axis=0) + 1) / (n_perm + 1)
        p_lo = ((null <= observed[None]).sum(axis=0) + 1) / (n_perm + 1)
        pval = np.minimum(2 * np.minimum(p_hi, p_lo), 1.0)
    elif alternative == 'greater':
        pval = ((null >= observed[None]).sum(axis=0) + 1) / (n_perm + 1)
    else:
        pval = ((null <= observed[None]).sum(axis=0) + 1) / (n_perm + 1)
    qval = benjamini_hochberg(pval)

    return dict(
        types=types,
        observed_fraction=pd.DataFrame(observed, index=types, columns=types),
        expected_fraction=pd.DataFrame(expected, index=types, columns=types),
        log2_ratio=pd.DataFrame(np.log2((observed + 1e-3) / (expected + 1e-3)),
                                index=types, columns=types),
        zscore=pd.DataFrame(zscore, index=types, columns=types),
        pval=pd.DataFrame(pval, index=types, columns=types),
        qval=pd.DataFrame(qval, index=types, columns=types),
    )


In [ ]:
positions = a.obsm['spatial']
labels_fine = a.obs['celltype_fine'].astype(str).values

res_fine = neighbourhood_enrichment(positions, labels_fine, K=K_NEIGHBOUR,
                                    n_perm=N_PERM, seed=SEED, alternative='two-sided')

# Read off the IgG_producing row, the question we care about.
row = pd.DataFrame({
    'observed': res_fine['observed_fraction'].loc[PLASMA_LABEL],
    'expected': res_fine['expected_fraction'].loc[PLASMA_LABEL],
    'log2':     res_fine['log2_ratio'].loc[PLASMA_LABEL],
    'zscore':   res_fine['zscore'].loc[PLASMA_LABEL],
    'p_2sided': res_fine['pval'].loc[PLASMA_LABEL],
    'q_BH_FDR': res_fine['qval'].loc[PLASMA_LABEL],
}).sort_values('zscore', ascending=False)
print('IgG_producing neighbour stats (sorted by z-score):')
row.round(3)


**How to read this table.** Each row is a candidate neighbour type. `observed`
is the fraction of an IgG-producing cell's K=10 neighbours that are this type,
`expected` is the fraction under random label shuffling, `log2` is the
enrichment, `zscore` and `q_BH_FDR` come from 500 permutations with BH-FDR
across the full label matrix. Positive enrichment with q<0.05 = the IgG cluster
preferentially neighbours this type. Negative = avoidance.

The biological question collapses to which `fib_*` row has the strongest
positive enrichment.


In [ ]:
# Heatmap of the full log2 enrichment matrix, masked by q<0.05, so the
# fibroblast subtype landscape is visible at a glance.
import matplotlib.colors as mcolors

log2 = res_fine['log2_ratio']
qval = res_fine['qval']
sig = qval < ALPHA
masked = log2.where(sig, np.nan)

fig, ax = plt.subplots(figsize=(8, 7))
vmax = float(np.nanmax(np.abs(masked.values))) if sig.any().any() else 1.0
im = ax.imshow(masked.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(log2.columns))); ax.set_xticklabels(log2.columns, rotation=45, ha='right')
ax.set_yticks(range(len(log2.index)));   ax.set_yticklabels(log2.index)
ax.set_title(f'log2(observed/expected) neighbour enrichment, masked q<{ALPHA}')
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.savefig(FIG / 'fib_neighbourhood_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()


## 7. Save outputs

In [ ]:
# Save the fibroblast-subset AnnData for downstream work (UMAP, scores,
# sub-labels travel with it).
fib.write_h5ad(OUT / 'adata_fibroblast.h5ad')
print('wrote', OUT / 'adata_fibroblast.h5ad', '->', fib.shape)

# Update the strip-wide AnnData with celltype_fine and re-save in place.
a.write_h5ad(OUT / 'adata_annotated.h5ad')
print('updated', OUT / 'adata_annotated.h5ad', '-> added celltype_fine')

# Numerical results so I do not need to re-run permutations.
row.to_csv(OUT / 'igg_neighbour_stats_fine.csv')
res_fine['log2_ratio'].to_csv(OUT / 'fine_log2_enrichment.csv')
res_fine['qval'].to_csv(OUT / 'fine_qval.csv')
panel_table.to_csv(OUT / 'fibroblast_panel_scores.csv')
print('wrote CSVs to', OUT)


## Summary

- I re-embedded the 4140 fibroblasts on their own HVGs / PCs to get a state
  axis driven by fibroblast biology, not by epidermal or immune contamination.
- I scored each cell against three keloid-relevant panels (myofibroblast, TGFb,
  hypoxia) and assigned each sub-cluster either a fibrogenic sub-label
  (`fib_myofibroblast` / `fib_tgfb` / `fib_hypoxia`) or `fibroblast_other` when
  no panel score was above its cross-cluster mean.
- I rebuilt `celltype_fine` for the whole strip and re-ran the same permutation
  neighbourhood test as notebook 04.
- The IgG-producing row of the enrichment table answers whether the
  immune-fibrosis circuit hypothesis is supported: a positive, FDR-significant
  enrichment with a specific `fib_*` subtype = yes, the IgG cluster is
  preferentially adjacent to that fibrogenic state.
- Outputs (`adata_fibroblast.h5ad`, updated `adata_annotated.h5ad`, CSVs of
  panel scores + neighbourhood stats, figures) are saved under
  `outputs/strip_01/`.

Next options: (a) repeat on strip 2 to check reproducibility, (b) look at
gene-level interaction candidates (ligand-receptor between IgG cells and the
top fibroblast subtype), (c) tissue-zone annotation at lambda=0.8 to layer
domains on top of cell types.
